# Case 3: Food Delivery Demand Pulse
## Demand Pattern Analysis & Surge Pricing Policy Recommendations

**Author:** Bhoumik Parmar  
**Date:** May 2026  
**Context:** A regional food-delivery company suspects peak demand is more nuanced than current surge rules. The Ops Head wants a one-day investigation: when does demand really spike, where, and what should the rider-incentive policy look like next month?

---

### Analysis Structure
1. Data Loading & Quality Checks
2. Demand Pattern Discovery (Hour × Day × City × Cuisine)
3. Surge Pricing Mismatch Analysis
4. 7-Day Demand Forecast
5. Three Policy Recommendations with Quantified Impact

### Assumptions
- Surge premium per order: ₹20 (industry average incremental rider incentive)
- Rider capacity: ~3 orders/hour (used for staffing estimates)
- Surge applied = 1 means the order was placed during a surge window; 0 = normal pricing
- No external data (weather, holidays, events) is used — noted as a limitation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

## 1. Data Loading & Quality Checks
Before any analysis, validate: nulls, duplicates, data types, date range, and value distributions.

In [ ]:
df = pd.read_csv('case3_food_delivery_orders.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f'Shape: {df.shape}')
print(f'\nDate range: {df["timestamp"].min()} → {df["timestamp"].max()} ({(df["timestamp"].max() - df["timestamp"].min()).days} days)')
print(f'\nNull counts:\n{df.isnull().sum()}')
print(f'\nDuplicate order_ids: {df["order_id"].duplicated().sum()}')
print(f'Fully duplicate rows: {df.duplicated().sum()}')
print(f'\nData types:\n{df.dtypes}')

In [ ]:
# Data quality: check for gaps in the date range
date_range = pd.date_range(df['timestamp'].dt.date.min(), df['timestamp'].dt.date.max())
actual_dates = pd.to_datetime(df['timestamp'].dt.date.unique())
missing_dates = set(date_range) - set(actual_dates)
print(f'Expected dates: {len(date_range)}')
print(f'Actual dates with data: {len(actual_dates)}')
print(f'Missing dates: {len(missing_dates)}')
if missing_dates:
    print(f'Missing: {sorted(missing_dates)}')
else:
    print('✅ No missing dates — data is complete for the 90-day window.')

In [ ]:
# Feature engineering
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['dow'] = df['timestamp'].dt.dayofweek  # 0=Mon
df['dow_name'] = df['timestamp'].dt.day_name()
df['is_weekend'] = df['dow'].isin([5, 6])
df['week'] = df['timestamp'].dt.isocalendar().week.astype(int)
df['month'] = df['timestamp'].dt.month
df['is_peak'] = df['hour'].isin([12, 13, 19, 20, 21])

# Summary stats
print('=== ORDER VALUE ===')
print(df['order_value'].describe().round(1))
print(f'\np95: ₹{df["order_value"].quantile(0.95):.0f}')
print(f'p99: ₹{df["order_value"].quantile(0.99):.0f}')

print(f'\n=== DELIVERY TIME ===')
print(df['delivery_time_min'].describe().round(1))
print(f'\np95: {df["delivery_time_min"].quantile(0.95):.0f} min')

print(f'\n=== SURGE ===')
print(f'Surge rate: {df["surge_applied"].mean():.1%} ({df["surge_applied"].sum():,} of {len(df):,} orders)')

print(f'\n=== CITIES ({df["city"].nunique()}) ===')
print(df['city'].value_counts())

print(f'\n=== CUISINES ({df["cuisine"].nunique()}) ===')
print(df['cuisine'].value_counts())

print(f'\n=== RESTAURANTS ===')
print(f'Unique: {df["restaurant_id"].nunique()}')

**Data Quality Verdict:** The dataset is clean — no nulls, no duplicates, no missing dates, complete 90-day coverage. All 50,000 orders have valid timestamps, cities, and values. Safe to proceed without imputation or cleaning.

## 2. Demand Pattern Discovery
The Ops Head's core question: *When does demand really spike?* Let's look at hour of day, day of week, city, and cuisine — in that order of importance.

In [ ]:
# 2.1 Hourly demand pattern — the most important view
hourly = df.groupby('hour').agg(
    orders=('order_id', 'count'),
    avg_order_value=('order_value', 'mean'),
    avg_delivery=('delivery_time_min', 'mean'),
    surge_rate=('surge_applied', 'mean')
).reset_index()
hourly['demand_share'] = hourly['orders'] / hourly['orders'].sum() * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Orders by hour
colors = ['#dc3545' if h in [12, 13, 19, 20, 21] else '#0d6efd' for h in range(24)]
axes[0].bar(hourly['hour'], hourly['orders'], color=colors, alpha=0.8)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Total Orders (90 days)')
axes[0].set_title('Demand by Hour — Two Clear Peaks')
axes[0].set_xticks(range(0, 24, 2))

# Delivery time by hour
axes[1].plot(hourly['hour'], hourly['avg_delivery'], color='#dc3545', linewidth=2, marker='o', markersize=4)
axes[1].axhline(y=hourly['avg_delivery'].mean(), color='gray', linestyle='--', alpha=0.5, label='Overall avg')
axes[1].fill_between([11.5, 13.5], 30, 50, alpha=0.1, color='orange')
axes[1].fill_between([18.5, 21.5], 30, 50, alpha=0.1, color='orange')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Avg Delivery Time (min)')
axes[1].set_title('Delivery Times Spike at Peak')
axes[1].set_xticks(range(0, 24, 2))
axes[1].legend()

# Surge rate by hour
axes[2].bar(hourly['hour'], hourly['surge_rate'] * 100, 
            color=['#ffc107' if h in [12, 13, 19, 20, 21] else '#adb5bd' for h in range(24)], alpha=0.8)
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Surge Rate (%)')
axes[2].set_title('Surge Concentrated at Peak Hours')
axes[2].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig('hourly_demand_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPeak hours (12-1pm, 7-9pm) account for '
      f'{hourly[hourly["hour"].isin([12,13,19,20,21])]["demand_share"].sum():.1f}% of total demand.')
print(f'Peak delivery time: {df[df["is_peak"]]["delivery_time_min"].mean():.1f} min')
print(f'Off-peak delivery time: {df[~df["is_peak"]]["delivery_time_min"].mean():.1f} min')
print(f'Difference: +{df[df["is_peak"]]["delivery_time_min"].mean() - df[~df["is_peak"]]["delivery_time_min"].mean():.1f} min at peak')

In [ ]:
# 2.2 Demand heatmap: Day of Week × Hour
hour_dow = df.groupby(['dow_name', 'hour']).size().reset_index(name='orders')
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hour_dow['dow_name'] = pd.Categorical(hour_dow['dow_name'], categories=dow_order, ordered=True)
pivot = hour_dow.pivot(index='dow_name', columns='hour', values='orders').fillna(0)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, fmt='.0f', linewidths=0.5, ax=ax)
ax.set_title('Demand Heatmap: Day of Week × Hour of Day', fontsize=14)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('demand_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key observation: The two-peak pattern (lunch + dinner) is consistent across ALL 7 days.')
print('No single day deviates meaningfully from the overall pattern.')

In [ ]:
# 2.3 City-level patterns — do cities behave differently?
city_hourly = df.groupby(['city', 'hour']).size().reset_index(name='orders')
n_days_city = df.groupby('city')['date'].nunique().reset_index()
n_days_city.columns = ['city', 'n_days']
city_hourly = city_hourly.merge(n_days_city, on='city')
city_hourly['orders_per_day'] = city_hourly['orders'] / city_hourly['n_days']

fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharey=False)
axes = axes.flatten()
cities = sorted(df['city'].unique())
for i, city in enumerate(cities):
    ch = city_hourly[city_hourly['city'] == city]
    axes[i].bar(ch['hour'], ch['orders_per_day'], 
                color=['#dc3545' if h in [12, 13, 19, 20, 21] else '#0d6efd' for h in ch['hour']], alpha=0.8)
    axes[i].set_title(f'{city}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Hour')
    axes[i].set_xticks(range(0, 24, 4))
    if i == 0 or i == 4:
        axes[i].set_ylabel('Orders/Day')
axes[7].axis('off')  # 7 cities, 8 subplots
plt.suptitle('Hourly Demand by City — Same Pattern, Different Scale', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('city_hourly_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print('All 7 cities show the same lunch + dinner peak shape.')
print('The difference is in magnitude, not timing:')
for city in cities:
    total = df[df['city'] == city].shape[0]
    print(f'  {city}: {total:,} orders ({total/len(df)*100:.1f}% share)')

In [ ]:
# 2.4 Weekend vs Weekday — the critical comparison
n_wkday = df[~df['is_weekend']]['date'].nunique()
n_wknd = df[df['is_weekend']]['date'].nunique()

wk_compare = df.groupby(['is_weekend', 'hour']).agg(
    orders=('order_id', 'count'),
    surge_rate=('surge_applied', 'mean'),
    avg_delivery=('delivery_time_min', 'mean')
).reset_index()

wk_compare['orders_per_day'] = wk_compare.apply(
    lambda r: r['orders'] / n_wknd if r['is_weekend'] else r['orders'] / n_wkday, axis=1
)
wk_compare['type'] = wk_compare['is_weekend'].map({False: 'Weekday', True: 'Weekend'})

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Demand comparison
for t, color in [('Weekday', '#0d6efd'), ('Weekend', '#dc3545')]:
    sub = wk_compare[wk_compare['type'] == t]
    axes[0].plot(sub['hour'], sub['orders_per_day'], label=t, color=color, linewidth=2)
axes[0].set_title('Demand: Nearly Identical Volume', fontsize=13)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Orders per Day')
axes[0].legend()
axes[0].set_xticks(range(0, 24, 2))

# Surge comparison
for t, color in [('Weekday', '#0d6efd'), ('Weekend', '#dc3545')]:
    sub = wk_compare[wk_compare['type'] == t]
    axes[1].plot(sub['hour'], sub['surge_rate'] * 100, label=t, color=color, linewidth=2)
axes[1].set_title('Surge Rate: Massive Weekend Over-Pricing', fontsize=13)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Surge Rate (%)')
axes[1].legend()
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig('weekend_vs_weekday.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n⚠️  THIS IS THE HEADLINE FINDING.')
print(f'Weekend dinner surge: {df[(df["is_weekend"]) & (df["hour"].isin([19,20,21]))]["surge_applied"].mean():.0%}')
print(f'Weekday dinner surge: {df[(~df["is_weekend"]) & (df["hour"].isin([19,20,21]))]["surge_applied"].mean():.0%}')
print(f'But demand per day is almost identical at dinner peak.')

## 3. Surge Pricing Mismatch Analysis

The data reveals three distinct surge inefficiencies:
1. **Off-peak surge waste** — 5.4% surge rate during hours with no supply constraint
2. **Weekend dinner over-surge** — 72% vs 45% weekday, same demand volume
3. **Pre-dinner blind spot** — reactive surge at 7pm instead of predictive pre-positioning

In [ ]:
# 3.1 Quantify each inefficiency
SURGE_PREMIUM = 20  # ₹ extra per surged order (industry avg)

# Off-peak waste
off_peak_hours = [h for h in range(24) if h not in [12, 13, 19, 20, 21]]
offpeak = df[df['hour'].isin(off_peak_hours)]
offpeak_surged = offpeak['surge_applied'].sum()

print('=== INEFFICIENCY 1: OFF-PEAK SURGE WASTE ===')
print(f'Off-peak hours: {len(off_peak_hours)} of 24')
print(f'Off-peak orders: {len(offpeak):,}')
print(f'Off-peak surge rate: {offpeak["surge_applied"].mean():.1%}')
print(f'Surged orders in off-peak: {offpeak_surged:,}')
print(f'Quarterly waste: ₹{offpeak_surged * SURGE_PREMIUM:,}')
print(f'Annualised waste: ₹{offpeak_surged * SURGE_PREMIUM * 4:,}')

print('\n=== INEFFICIENCY 2: WEEKEND DINNER OVER-SURGE ===')
wkday_dinner = df[(~df['is_weekend']) & (df['hour'].isin([19, 20, 21]))]
wknd_dinner = df[(df['is_weekend']) & (df['hour'].isin([19, 20, 21]))]
excess_surged = int(wknd_dinner['surge_applied'].sum() - len(wknd_dinner) * wkday_dinner['surge_applied'].mean())
print(f'Weekday dinner: {len(wkday_dinner):,} orders, {wkday_dinner["surge_applied"].mean():.0%} surge')
print(f'Weekend dinner: {len(wknd_dinner):,} orders, {wknd_dinner["surge_applied"].mean():.0%} surge')
print(f'Excess surged orders (vs weekday rate): {excess_surged:,}')
print(f'Quarterly waste: ₹{excess_surged * SURGE_PREMIUM:,}')
print(f'Annualised waste: ₹{excess_surged * SURGE_PREMIUM * 4:,}')

print('\n=== INEFFICIENCY 3: PRE-DINNER BLIND SPOT ===')
h18 = df[df['hour'] == 18]
h19 = df[df['hour'] == 19]
print(f'Hour 18 → 19 demand jump: +{(len(h19)-len(h18))/len(h18)*100:.0f}%')
print(f'Hour 18 → 19 surge jump: {h18["surge_applied"].mean():.1%} → {h19["surge_applied"].mean():.1%}')
print(f'Hour 18 → 19 delivery time: {h18["delivery_time_min"].mean():.1f} → {h19["delivery_time_min"].mean():.1f} min')
print(f'\nIf pre-positioning reduces dinner surge by 12pp (52% → 40%):')
dinner_orders_quarter = len(df[df['hour'].isin([19, 20, 21])])
saved_surges = int(dinner_orders_quarter * 0.12)
print(f'  Avoided surged orders/quarter: ~{saved_surges:,}')
print(f'  Quarterly savings (net of pre-position cost): ~₹{saved_surges * 8:,} (at ₹8 net saving/order)')
print(f'  Annualised savings: ~₹{saved_surges * 8 * 4:,}')

In [ ]:
# 3.2 Visualise the surge-demand mismatch
fig, ax = plt.subplots(figsize=(14, 6))

# Normalise both to 0-1 for comparison
hourly_norm = hourly.copy()
hourly_norm['demand_norm'] = hourly_norm['demand_share'] / hourly_norm['demand_share'].max()
hourly_norm['surge_norm'] = hourly_norm['surge_rate'] / hourly_norm['surge_rate'].max()

ax.fill_between(hourly_norm['hour'], hourly_norm['demand_norm'], alpha=0.3, color='#0d6efd', label='Demand (normalised)')
ax.plot(hourly_norm['hour'], hourly_norm['demand_norm'], color='#0d6efd', linewidth=2)
ax.plot(hourly_norm['hour'], hourly_norm['surge_norm'], color='#dc3545', linewidth=2.5, marker='o', markersize=5, label='Surge Rate (normalised)')

# Annotate the mismatch zones
ax.annotate('Off-peak surge\n(waste)', xy=(3, 0.12), fontsize=10, color='#dc3545', fontweight='bold',
            ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='#fce4ec', alpha=0.8))
ax.annotate('Pre-dinner gap\n(blind spot)', xy=(17.5, 0.15), fontsize=10, color='#dc3545', fontweight='bold',
            ha='center', bbox=dict(boxstyle='round,pad=0.3', facecolor='#fce4ec', alpha=0.8))

ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Normalised Value (0–1)', fontsize=12)
ax.set_title('Surge vs Demand: Where They Diverge Is Where Money Leaks', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig('surge_demand_mismatch.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 7-Day Demand Forecast

**Methodology:** Trend + Day-of-Week Seasonal Decomposition
- Extract linear trend from 90-day daily order counts
- Compute day-of-week seasonal effects (average deviation from trend for each weekday)
- Forecast = Trend + Seasonal + 95% CI from residual standard deviation

**Why this model:** The data shows no strong growth trend and stable weekly patterns. A simple decomposition is interpretable, robust, and appropriate for a 7-day horizon. For longer horizons or when external factors (weather, holidays, campaigns) are available, I'd recommend Prophet or SARIMAX.

In [ ]:
# Build the forecast
daily = df.groupby('date').agg(orders=('order_id', 'count'), revenue=('order_value', 'sum')).reset_index()
daily['date'] = pd.to_datetime(daily['date'])
daily['dow'] = daily['date'].dt.dayofweek

# Trend
x = np.arange(len(daily))
coeffs = np.polyfit(x, daily['orders'].values, 1)
trend = np.polyval(coeffs, x)
print(f'Trend: {coeffs[0]:.3f} orders/day (essentially flat)')

# Seasonal
detrended = daily['orders'].values - trend
dow_effect = pd.Series(detrended).groupby(daily['dow']).mean()
print(f'\nDay-of-week effects:')
for d, v in dow_effect.items():
    print(f'  {["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][d]}: {v:+.1f} orders')

# Residual
seasonal = daily['dow'].map(dow_effect).values
residual = detrended - seasonal
residual_std = np.std(residual)
print(f'\nResidual std: {residual_std:.1f} orders')
print(f'95% CI width: ±{1.96 * residual_std:.0f} orders')

# Fitted values
daily['trend'] = trend
daily['fitted'] = trend + seasonal

# Forecast
last_date = daily['date'].max()
forecast_dates = pd.date_range(last_date + timedelta(days=1), periods=7)
forecast_x = np.arange(len(daily), len(daily) + 7)
forecast_trend = np.polyval(coeffs, forecast_x)
forecast_dow = np.array([dow_effect[d.dayofweek] for d in forecast_dates])
forecast_orders = forecast_trend + forecast_dow
ci_lower = forecast_orders - 1.96 * residual_std
ci_upper = forecast_orders + 1.96 * residual_std

print(f'\n=== 7-DAY FORECAST ===')
for i, d in enumerate(forecast_dates):
    print(f'  {d.strftime("%Y-%m-%d")} ({d.day_name()[:3]}): '
          f'{forecast_orders[i]:.0f} orders [{ci_lower[i]:.0f} – {ci_upper[i]:.0f}]')

In [ ]:
# Forecast visualisation
fig, ax = plt.subplots(figsize=(16, 5))

# Historical
ax.plot(daily['date'], daily['orders'], color='#adb5bd', alpha=0.5, linewidth=1, label='Actual')
ax.plot(daily['date'], daily['fitted'], color='#0d6efd', linewidth=1.5, linestyle='--', label='Fitted')

# Forecast
ax.plot(forecast_dates, forecast_orders, color='#dc3545', linewidth=2.5, marker='o', markersize=6, label='Forecast')
ax.fill_between(forecast_dates, ci_lower, ci_upper, alpha=0.2, color='#dc3545', label='95% CI')

# Divider
ax.axvline(x=last_date, color='gray', linestyle=':', alpha=0.5)
ax.text(last_date, ax.get_ylim()[1], ' ← Historical | Forecast →', fontsize=9, color='gray', va='top')

ax.set_xlabel('Date')
ax.set_ylabel('Daily Orders')
ax.set_title('90-Day Historical + 7-Day Forecast', fontsize=14)
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig('forecast.png', dpi=150, bbox_inches='tight')
plt.show()

# Save forecast as CSV
forecast_df = pd.DataFrame({
    'date': forecast_dates,
    'predicted_orders': forecast_orders.round(0).astype(int),
    'ci_lower': ci_lower.round(0).astype(int),
    'ci_upper': ci_upper.round(0).astype(int)
})
forecast_df.to_csv('forecast_output.csv', index=False)
print('Forecast saved to forecast_output.csv')

### Forecast Evaluation Note

In production, I would evaluate this forecast using:
1. **MAPE** (Mean Absolute Percentage Error) — target <5% for a 7-day horizon on stable demand
2. **Coverage** — 95% CI should contain actual values ≥90% of the time
3. **Backtest** — hold out the last 7 days, forecast from day 83, compare to actuals

The current model is intentionally simple because the data shows **no strong trend** and **stable day-of-week patterns**. Adding complexity (ARIMA, Prophet) would not materially improve a 7-day forecast on this data. If seasonal events (Diwali, IPL matches, monsoon) enter the picture, those covariates would need to be added.

## 5. Three Policy Recommendations

Each recommendation is:
- Tied to a specific data finding
- Quantified with expected impact
- Accompanied by edge cases and risks
- Actionable by the Ops Head on Monday morning

In [ ]:
# Summary impact table
impact = pd.DataFrame({
    'Recommendation': [
        '1. Eliminate off-peak surge',
        '2. Cap weekend dinner surge at weekday levels',
        '3. Pre-position riders at 5-6pm for dinner ramp'
    ],
    'Data Finding': [
        '5.4% surge in off-peak hours with no supply constraint',
        '72% weekend vs 45% weekday surge, identical demand volume',
        '52% demand jump H18→H19, surge jumps 815% reactively'
    ],
    'Quarterly Savings': ['₹30,300', '₹22,260', '~₹60,000 (net)'],
    'Annualised Savings': ['₹1.2L', '₹89K', '~₹2.4L'],
    'Risk Level': ['None', 'Low', 'Medium'],
    'Implementation Time': ['1 day', '1 week', '4-6 weeks (A/B test)']
})
print(impact.to_string(index=False))

### Recommendation 1: Eliminate Off-Peak Surge Entirely

**Finding:** 5.4% of orders during off-peak hours trigger surge pricing. These hours have demand 3–7× lower than peak and delivery times at baseline (37 min). There is no supply constraint justifying surge.

**Action:** Set surge threshold to zero for hours outside 12–1pm and 6–9pm.

**Impact:** ₹30,300/quarter saved, ₹1.2L annualised. Zero operational risk.

**Edge case:** Late-night orders during major events (NYE, IPL) in metro cities. Build a manual override list for known high-demand dates.

---

### Recommendation 2: Cap Weekend Dinner Surge at Weekday Levels

**Finding:** Weekend dinner surge is 72% vs weekday 45%, but order volume per day is nearly identical (~160 orders at dinner peak). The company is paying 60% more in surge during weekends for the same demand.

**Action:** (1) Immediate — cap weekend dinner surge at 50%. (2) Next month — introduce a flat weekend shift bonus (₹200–300/day) to improve weekend rider supply structurally.

**Impact:** ₹22,260/quarter from cap alone, ₹89K annualised. Shift bonus costs ~₹3.25L/year but is expected to reduce weekend surge below 40%, creating net savings.

**Edge case:** Monitor weekend p95 delivery times after the cap. If >70 min, raise cap to 55% and increase shift bonus.

---

### Recommendation 3: Pre-Position Riders at 5–6pm for Dinner

**Finding:** Demand jumps 52% from 6pm to 7pm. Surge rate jumps from 5.7% to 52.1% — the system is purely reactive. Delivery times jump +6 min at peak. Pre-positioning supply before the predictable dinner ramp would reduce both surge costs and delivery times.

**Action:** Use hour 17–18 order velocity to trigger a pre-position notification to off-duty riders, offering a dinner bonus (₹10–15/order, cheaper than ₹20 surge). Target: reduce dinner surge from 52% to 40%.

**Impact:** ~₹60K/quarter net savings (₹2.4L annualised). P95 delivery time target: <60 min (currently 65).

**Validation:** A/B test in Bangalore (highest volume) for 4 weeks. Alternating-day design (supply-side intervention, not user-level). Primary metric: dinner surge rate. Guardrail: rider earnings, p95 delivery.

## Appendix: Limitations & Next Steps

### Limitations
1. **No supply data.** The dataset contains orders only — no rider availability, online rider counts, or acceptance rates. The surge analysis infers supply constraints from delivery times, but direct supply data would strengthen the recommendations.
2. **No external covariates.** Weather, public holidays, local events (IPL matches), and marketing campaigns all affect demand. The forecast model does not account for these.
3. **Surge premium assumption.** ₹20/order is an industry average. The actual company's surge economics may differ — recommendations should be re-calibrated with real rider incentive data.
4. **90 days of data.** The dataset covers January–March (winter/spring). Summer and monsoon patterns may differ. Recommendations should be re-validated quarterly.

### Next Steps
1. Obtain rider supply data (online riders per hour per city) to validate the supply-side interpretation
2. Add weather and holiday covariates to the forecast model
3. Build an anomaly detection system for daily order counts (alert when actuals deviate >2σ from forecast)
4. Instrument A/B test for Recommendation 3 in Bangalore
5. Build a city-level surge policy engine that uses real-time demand/supply ratio rather than fixed rules